In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder_path = '/content/drive/MyDrive/STOR566Project'
print("Files in shared folder:")
print(os.listdir(folder_path))

# data_augmentations + dataset + train/val split

import math
import json
import h5py
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from scipy.ndimage import gaussian_filter1d


def gauss_smooth(inputs, device, smooth_kernel_std=2, smooth_kernel_size=100, padding='same'):
    inp = np.zeros(smooth_kernel_size, dtype=np.float32)
    inp[smooth_kernel_size // 2] = 1
    gaussKernel = gaussian_filter1d(inp, smooth_kernel_std)
    validIdx = np.argwhere(gaussKernel > 0.01)
    gaussKernel = gaussKernel[validIdx]
    gaussKernel = np.squeeze(gaussKernel / np.sum(gaussKernel))

    gaussKernel = torch.tensor(gaussKernel, dtype=torch.float32, device=device)
    gaussKernel = gaussKernel.view(1, 1, -1)

    B, T, C = inputs.shape
    inputs = inputs.permute(0, 2, 1)
    gaussKernel = gaussKernel.repeat(C, 1, 1)

    smoothed = F.conv1d(inputs, gaussKernel, padding=padding, groups=C)
    return smoothed.permute(0, 2, 1)


class BrainToTextDataset(Dataset):
    def __init__(
        self,
        trial_indicies,
        n_batches,
        split='train',
        batch_size=64,
        days_per_batch=1,
        random_seed=-1,
        must_include_days=None,
        feature_subset=None,
    ):
        if random_seed != -1:
            np.random.seed(random_seed)
            torch.manual_seed(random_seed)

        if split not in ["train", "test"]:
            raise ValueError(f'split must be "train" or "test", got {split}')

        self.split = split
        self.days_per_batch = days_per_batch
        self.batch_size = batch_size
        self.n_batches = n_batches

        self.trial_indicies = trial_indicies
        self.n_days = len(trial_indicies.keys())
        self.feature_subset = feature_subset
        self.n_trials = sum(len(trial_indicies[d]["trials"]) for d in trial_indicies)

        if must_include_days is not None and len(must_include_days) > days_per_batch:
            raise ValueError("must_include_days must be <= days_per_batch")

        if must_include_days is not None and len(must_include_days) > self.n_days and split != "train":
            raise ValueError("must_include_days not valid for test data")

        if must_include_days is not None:
            for i, d in enumerate(must_include_days):
                if d < 0:
                    must_include_days[i] = self.n_days + d

        self.must_include_days = must_include_days

        if self.split == "train" and self.days_per_batch > self.n_days:
            raise ValueError(f"days_per_batch {days_per_batch} > n_days {self.n_days}")

        if self.split == "train":
            self.batch_index = self.create_batch_index_train()
        else:
            self.batch_index = self.create_batch_index_test()
            self.n_batches = len(self.batch_index.keys())

    def __len__(self):
        return self.n_batches

    def __getitem__(self, idx):
        batch = {
            "input_features": [],
            "seq_class_ids": [],
            "n_time_steps": [],
            "phone_seq_lens": [],
            "day_indicies": [],
            "transcriptions": [],
            "block_nums": [],
            "trial_nums": [],
        }

        index = self.batch_index[idx]

        for d in index.keys():
            with h5py.File(self.trial_indicies[d]["session_path"], "r") as f:
                for t in index[d]:
                    try:
                        g = f[f"trial_{t:04d}"]

                        x = torch.from_numpy(g["input_features"][:])
                        if self.feature_subset:
                            x = x[:, self.feature_subset]

                        batch["input_features"].append(x)
                        batch["seq_class_ids"].append(torch.from_numpy(g["seq_class_ids"][:]))
                        batch["transcriptions"].append(torch.from_numpy(g["transcription"][:]))
                        batch["n_time_steps"].append(g.attrs["n_time_steps"])
                        batch["phone_seq_lens"].append(g.attrs["seq_len"])
                        batch["day_indicies"].append(int(d))
                        batch["block_nums"].append(g.attrs["block_num"])
                        batch["trial_nums"].append(g.attrs["trial_num"])
                    except Exception as e:
                        print(f"Error loading trial {t} from {self.trial_indicies[d]['session_path']}: {e}")
                        continue

        batch["input_features"] = pad_sequence(batch["input_features"], batch_first=True, padding_value=0)
        batch["seq_class_ids"] = pad_sequence(batch["seq_class_ids"], batch_first=True, padding_value=0)

        batch["n_time_steps"] = torch.tensor(batch["n_time_steps"])
        batch["phone_seq_lens"] = torch.tensor(batch["phone_seq_lens"])
        batch["day_indicies"] = torch.tensor(batch["day_indicies"])
        batch["transcriptions"] = torch.stack(batch["transcriptions"])
        batch["block_nums"] = torch.tensor(batch["block_nums"])
        batch["trial_nums"] = torch.tensor(batch["trial_nums"])

        return batch

    def create_batch_index_train(self):
        batch_index = {}

        if self.must_include_days is not None:
            non_must_include_days = [d for d in self.trial_indicies.keys() if d not in self.must_include_days]

        for batch_idx in range(self.n_batches):
            batch = {}

            if self.must_include_days is not None and len(self.must_include_days) > 0:
                days = np.concatenate(
                    (
                        self.must_include_days,
                        np.random.choice(
                            non_must_include_days,
                            size=self.days_per_batch - len(self.must_include_days),
                            replace=False,
                        ),
                    )
                )
            else:
                days = np.random.choice(list(self.trial_indicies.keys()), size=self.days_per_batch, replace=False)

            num_trials = math.ceil(self.batch_size / self.days_per_batch)

            for d in days:
                trial_idxs = np.random.choice(self.trial_indicies[d]["trials"], size=num_trials, replace=True)
                batch[d] = trial_idxs

            extra_trials = (num_trials * len(days)) - self.batch_size
            while extra_trials > 0:
                d = np.random.choice(days)
                batch[d] = batch[d][:-1]
                extra_trials -= 1

            batch_index[batch_idx] = batch

        return batch_index

    def create_batch_index_test(self):
        batch_index = {}
        batch_idx = 0

        for d in self.trial_indicies.keys():
            num_trials = len(self.trial_indicies[d]["trials"])
            num_batches = (num_trials + self.batch_size - 1) // self.batch_size

            for i in range(num_batches):
                start_idx = i * self.batch_size
                end_idx = min((i + 1) * self.batch_size, num_trials)
                batch_trials = self.trial_indicies[d]["trials"][start_idx:end_idx]
                batch_index[batch_idx] = {d: batch_trials}
                batch_idx += 1

        return batch_index


def train_test_split_indicies(file_paths, test_percentage=0.1, seed=-1, bad_trials_dict=None):
    if seed != -1:
        np.random.seed(seed)

    trials_per_day = {}
    for i, path in enumerate(file_paths):
        session = [s for s in path.split("/") if (s.startswith("t15.20") or s.startswith("t12.20"))][0]
        good_trial_indices = []

        if os.path.exists(path):
            with h5py.File(path, "r") as f:
                num_trials = len(list(f.keys()))
                for t in range(num_trials):
                    key = f"trial_{t:04d}"
                    block_num = f[key].attrs["block_num"]
                    trial_num = f[key].attrs["trial_num"]

                    if (
                        bad_trials_dict is not None
                        and session in bad_trials_dict
                        and str(block_num) in bad_trials_dict[session]
                        and trial_num in bad_trials_dict[session][str(block_num)]
                    ):
                        continue

                    good_trial_indices.append(t)

        trials_per_day[i] = {
            "num_trials": len(good_trial_indices),
            "trial_indices": good_trial_indices,
            "session_path": path,
        }

    train_trials, test_trials = {}, {}

    for day in trials_per_day.keys():
        num_trials = trials_per_day[day]["num_trials"]
        all_trial_indices = trials_per_day[day]["trial_indices"]
        session_path = trials_per_day[day]["session_path"]

        if test_percentage == 0:
            train_trials[day] = {"trials": all_trial_indices, "session_path": session_path}
            test_trials[day] = {"trials": [], "session_path": session_path}
            continue
        if test_percentage == 1:
            train_trials[day] = {"trials": [], "session_path": session_path}
            test_trials[day] = {"trials": all_trial_indices, "session_path": session_path}
            continue

        num_test = max(1, int(num_trials * test_percentage))
        test_indices = np.random.choice(all_trial_indices, size=num_test, replace=False).tolist()
        train_indices = [idx for idx in all_trial_indices if idx not in test_indices]

        train_trials[day] = {"trials": train_indices, "session_path": session_path}
        test_trials[day] = {"trials": test_indices, "session_path": session_path}

    return train_trials, test_trials


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in shared folder:
['data', 'models', '.ipynb_checkpoints']


In [5]:
!pip install braindecode mne

# CNNTransformer (EEGNet + FLAN-T5 encoder) + Trainer

import logging
import pathlib
import pickle
import random
import sys
import time

import torchaudio.functional as taF
from braindecode.models import EEGNet
from omegaconf import OmegaConf
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from transformers import T5ForConditionalGeneration, T5Tokenizer


class CNNTransformer(torch.nn.Module):
    def __init__(
        self,
        neural_dim,
        n_units,
        n_days,
        n_classes,
        rnn_dropout,
        input_dropout,
        n_layers,
        patch_size,
        patch_stride,
        eenet_model,
        transformer_model,
        temporal_bin,
    ):
        super().__init__()
        self.neural_dim = neural_dim
        self.n_units = n_units
        self.n_days = n_days
        self.n_classes = n_classes
        self.temporal_bin = temporal_bin

        self.eegnet = eenet_model
        self.transformer_model = transformer_model

        d_model = self.transformer_model.config.d_model

        self.proj_to_t5 = torch.nn.Linear(n_units, d_model)
        self.day_embeddings = torch.nn.Embedding(n_days, d_model)

        self.input_dropout = torch.nn.Dropout(input_dropout)
        self.rnn_dropout = torch.nn.Dropout(rnn_dropout)

        self.out = torch.nn.Linear(d_model, n_classes)

    def forward(self, features, day_indices):
        B, T, C = features.shape
        bin_len = self.temporal_bin

        S = (T + bin_len - 1) // bin_len
        T_eff = S * bin_len
        if T_eff > T:
            pad = torch.zeros(B, T_eff - T, C, device=features.device, dtype=features.dtype)
            x = torch.cat([features, pad], dim=1)
        else:
            x = features

        x = x.view(B, S, bin_len, C).permute(0, 1, 3, 2)
        x = x.reshape(B * S, C, bin_len)

        eeg_feat = self.eegnet(x)
        eeg_feat = eeg_feat.view(B, S, self.n_units)

        h = self.proj_to_t5(eeg_feat)

        day_emb = self.day_embeddings(day_indices).unsqueeze(1)
        h = h + day_emb

        h = self.input_dropout(h)

        attn_mask = torch.ones(B, S, dtype=torch.long, device=h.device)
        encoder_outputs = self.transformer_model.encoder(
            inputs_embeds=h,
            attention_mask=attn_mask,
        )
        hidden = encoder_outputs.last_hidden_state
        hidden = self.rnn_dropout(hidden)

        logits = self.out(hidden)
        return logits


class CNN_Transformer_Trainer:
    def __init__(self, args):
        self.args = args
        self.logger = None
        self.device = None
        self.model = None
        self.optimizer = None
        self.learning_rate_scheduler = None
        self.ctc_loss = None

        self.best_val_PER = torch.inf
        self.best_val_loss = torch.inf

        self.train_dataset = None
        self.val_dataset = None
        self.train_loader = None
        self.val_loader = None

        self.transform_args = self.args["dataset"]["data_transforms"]

        if args["mode"] == "train":
            os.makedirs(self.args["output_dir"], exist_ok=True)
        if (
            args["save_best_checkpoint"]
            or args["save_all_val_steps"]
            or args["save_final_model"]
        ):
            os.makedirs(self.args["checkpoint_dir"], exist_ok=True)

        self.logger = logging.getLogger("BrainToTextTrainer")
        for h in list(self.logger.handlers):
            self.logger.removeHandler(h)
        self.logger.setLevel(logging.INFO)
        formatter = logging.Formatter(fmt="%(asctime)s: %(message)s")

        if args["mode"] == "train":
            fh = logging.FileHandler(str(pathlib.Path(self.args["output_dir"], "training_log")))
            fh.setFormatter(formatter)
            self.logger.addHandler(fh)

        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(formatter)
        self.logger.addHandler(sh)

        if torch.cuda.is_available():
            gpu_num = self.args.get("gpu_number", 0)
            try:
                gpu_num = int(gpu_num)
            except ValueError:
                self.logger.warning(f"Invalid gpu_number value: {gpu_num}. Using 0 instead.")
                gpu_num = 0

            max_gpu_index = torch.cuda.device_count() - 1
            if gpu_num > max_gpu_index:
                self.logger.warning(
                    f"Requested GPU {gpu_num} not available. Using 0 instead."
                )
                gpu_num = 0

            try:
                self.device = torch.device(f"cuda:{gpu_num}")
                _ = torch.tensor([1.0]).to(self.device) * 2
            except Exception as e:
                self.logger.error(f"Error initializing CUDA device {gpu_num}: {str(e)}")
                self.logger.info("Falling back to CPU")
                self.device = torch.device("cpu")
        else:
            self.device = torch.device("cpu")

        self.logger.info(f"Using device: {self.device}")

        if self.args["seed"] != -1:
            np.random.seed(self.args["seed"])
            random.seed(self.args["seed"])
            torch.manual_seed(self.args["seed"])

        eenet = EEGNet(
            n_chans=self.args["model"]["n_input_features"],
            n_outputs=self.args["model"]["n_units"],
            n_times=self.args["dataset"]["temporal_bin"],
        )

        self.tokenizer = T5Tokenizer.from_pretrained(
            self.args["model"]["transformer_name"]
        )
        self.t5model = T5ForConditionalGeneration.from_pretrained(
            self.args["model"]["transformer_name"]
        )

        self.model = CNNTransformer(
            neural_dim=self.args["model"]["n_input_features"],
            n_units=self.args["model"]["n_units"],
            n_days=len(self.args["dataset"]["sessions"]),
            n_classes=self.args["dataset"]["n_classes"],
            rnn_dropout=self.args["model"]["rnn_dropout"],
            input_dropout=self.args["model"]["input_network"]["input_layer_dropout"],
            n_layers=self.args["model"]["n_layers"],
            patch_size=self.args["model"]["patch_size"],
            patch_stride=self.args["model"]["patch_stride"],
            eenet_model=eenet,
            transformer_model=self.t5model,
            temporal_bin=self.args["dataset"]["temporal_bin"],
        )

        if self.args["use_torch_compile"]:
            self.model = torch.compile(self.model)

        self.logger.info("Initialized CNN transformer model")
        self.logger.info(self.model)

        total_params = sum(p.numel() for p in self.model.parameters())
        self.logger.info(f"Model has {total_params:,} parameters")

        day_params = 0
        for name, param in self.model.named_parameters():
            if "day" in name:
                day_params += param.numel()
        self.logger.info(
            f"Model has {day_params:,} day-specific parameters "
            f"| {((day_params / total_params) * 100):.2f}% of total parameters"
        )

        train_file_paths = [
            os.path.join(self.args["dataset"]["dataset_dir"], s, "data_train.hdf5")
            for s in self.args["dataset"]["sessions"]
        ]
        val_file_paths = [
            os.path.join(self.args["dataset"]["dataset_dir"], s, "data_val.hdf5")
            for s in self.args["dataset"]["sessions"]
        ]

        if len(set(train_file_paths)) != len(train_file_paths):
            raise ValueError("Duplicate sessions listed in the train dataset")
        if len(set(val_file_paths)) != len(val_file_paths):
            raise ValueError("Duplicate sessions listed in the val dataset")

        train_trials, _ = train_test_split_indicies(
            file_paths=train_file_paths,
            test_percentage=0,
            seed=self.args["dataset"]["seed"],
            bad_trials_dict=self.args["dataset"].get("bad_trials_dict", None),
        )
        _, val_trials = train_test_split_indicies(
            file_paths=val_file_paths,
            test_percentage=1,
            seed=self.args["dataset"]["seed"],
            bad_trials_dict=self.args["dataset"].get("bad_trials_dict", None),
        )

        with open(os.path.join(self.args["output_dir"], "train_val_trials.json"), "w") as f:
            json.dump({"train": train_trials, "val": val_trials}, f)

        feature_subset = None
        if ("feature_subset" in self.args["dataset"]) and self.args["dataset"]["feature_subset"] is not None:
            feature_subset = self.args["dataset"]["feature_subset"]
            self.logger.info(f"Using only a subset of features: {feature_subset}")

        self.train_dataset = BrainToTextDataset(
            trial_indicies=train_trials,
            split="train",
            days_per_batch=self.args["dataset"]["days_per_batch"],
            n_batches=self.args["num_training_batches"],
            batch_size=self.args["dataset"]["batch_size"],
            must_include_days=None,
            random_seed=self.args["dataset"]["seed"],
            feature_subset=feature_subset,
        )
        self.train_loader = DataLoader(
            self.train_dataset,
            batch_size=None,
            shuffle=self.args["dataset"]["loader_shuffle"],
            num_workers=self.args["dataset"]["num_dataloader_workers"],
            pin_memory=True,
        )

        self.val_dataset = BrainToTextDataset(
            trial_indicies=val_trials,
            split="test",
            days_per_batch=None,
            n_batches=None,
            batch_size=self.args["dataset"]["batch_size"],
            must_include_days=None,
            random_seed=self.args["dataset"]["seed"],
            feature_subset=feature_subset,
        )
        self.val_loader = DataLoader(
            self.val_dataset,
            batch_size=None,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        )

        self.logger.info("Successfully initialized datasets")

        self.optimizer = self.create_optimizer()

        if self.args["lr_scheduler_type"] == "linear":
            self.learning_rate_scheduler = torch.optim.lr_scheduler.LinearLR(
                optimizer=self.optimizer,
                start_factor=1.0,
                end_factor=self.args["lr_min"] / self.args["lr_max"],
                total_iters=self.args["lr_decay_steps"],
            )
        elif self.args["lr_scheduler_type"] == "cosine":
            self.learning_rate_scheduler = self.create_cosine_lr_scheduler(self.optimizer)
        else:
            raise ValueError(f"Invalid lr_scheduler_type: {self.args['lr_scheduler_type']}")

        self.ctc_loss = torch.nn.CTCLoss(blank=0, reduction="none", zero_infinity=True)

        if self.args["init_from_checkpoint"] and self.args["init_checkpoint_path"]:
            self.load_model_checkpoint(self.args["init_checkpoint_path"])

        for name, param in self.model.named_parameters():
            if not self.args["model"]["rnn_trainable"] and "gru" in name:
                param.requires_grad = False
            elif (
                not self.args["model"]["input_network"]["input_trainable"]
                and "day" in name
            ):
                param.requires_grad = False

        self.model.to(self.device)

    def create_optimizer(self):
        bias_params = []
        day_params = []
        other_params = []

        for name, p in self.model.named_parameters():
            if not p.requires_grad:
                continue

            if "gru.bias" in name or "out.bias" in name:
                bias_params.append(p)
            elif "day" in name:
                day_params.append(p)
            else:
                other_params.append(p)

        param_groups = []
        if bias_params:
            param_groups.append(
                {"params": bias_params, "weight_decay": 0.0, "group_type": "bias"}
            )
        if day_params:
            param_groups.append(
                {
                    "params": day_params,
                    "lr": self.args["lr_max_day"],
                    "weight_decay": self.args["weight_decay_day"],
                    "group_type": "day_layer",
                }
            )
        if other_params:
            param_groups.append(
                {"params": other_params, "group_type": "other"}
            )

        try:
            optim = torch.optim.AdamW(
                param_groups,
                lr=self.args["lr_max"],
                betas=(self.args["beta0"], self.args["beta1"]),
                eps=self.args["epsilon"],
                weight_decay=self.args["weight_decay"],
                fused=True,
            )
        except TypeError:
            optim = torch.optim.AdamW(
                param_groups,
                lr=self.args["lr_max"],
                betas=(self.args["beta0"], self.args["beta1"]),
                eps=self.args["epsilon"],
                weight_decay=self.args["weight_decay"],
            )

        return optim

    def create_cosine_lr_scheduler(self, optim):
        lr_max = self.args["lr_max"]
        lr_min = self.args["lr_min"]
        lr_decay_steps = self.args["lr_decay_steps"]

        lr_max_day = self.args["lr_max_day"]
        lr_min_day = self.args["lr_min_day"]
        lr_decay_steps_day = self.args["lr_decay_steps_day"]

        lr_warmup_steps = self.args["lr_warmup_steps"]
        lr_warmup_steps_day = self.args["lr_warmup_steps_day"]

        def lr_lambda(current_step, min_lr_ratio, decay_steps, warmup_steps):
            if current_step < warmup_steps:
                return float(current_step) / float(max(1, warmup_steps))
            if current_step < decay_steps:
                progress = float(current_step - warmup_steps) / float(
                    max(1, decay_steps - warmup_steps)
                )
                cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
                return max(
                    min_lr_ratio, min_lr_ratio + (1 - min_lr_ratio) * cosine_decay
                )
            return min_lr_ratio

        if len(optim.param_groups) == 3:
            lr_lambdas = [
                lambda step: lr_lambda(step, lr_min / lr_max, lr_decay_steps, lr_warmup_steps),
                lambda step: lr_lambda(step, lr_min_day / lr_max_day, lr_decay_steps_day, lr_warmup_steps_day),
                lambda step: lr_lambda(step, lr_min / lr_max, lr_decay_steps, lr_warmup_steps),
            ]
        elif len(optim.param_groups) == 2:
            lr_lambdas = [
                lambda step: lr_lambda(step, lr_min / lr_max, lr_decay_steps, lr_warmup_steps),
                lambda step: lr_lambda(step, lr_min / lr_max, lr_decay_steps, lr_warmup_steps),
            ]
        else:
            raise ValueError(f"Unexpected number of param groups: {len(optim.param_groups)}")

        return LambdaLR(optim, lr_lambdas, -1)

    def load_model_checkpoint(self, load_path):
        checkpoint = torch.load(load_path, weights_only=False, map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        self.learning_rate_scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        self.best_val_PER = checkpoint.get("val_PER", torch.inf)
        self.best_val_loss = checkpoint.get("val_loss", torch.inf)

        self.model.to(self.device)
        for state in self.optimizer.state.values():
            for k, v in state.items():
                if isinstance(v, torch.Tensor):
                    state[k] = v.to(self.device)

        self.logger.info(f"Loaded model from checkpoint: {load_path}")

    def save_model_checkpoint(self, save_path, PER, loss):
        checkpoint = {
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.learning_rate_scheduler.state_dict(),
            "val_PER": PER,
            "val_loss": loss,
        }
        torch.save(checkpoint, save_path)
        self.logger.info(f"Saved model to checkpoint: {save_path}")

        args_save_path = os.path.join(self.args["checkpoint_dir"], "args.yaml")
        OmegaConf.save(config=self.args, f=args_save_path)

    def transform_data(self, features, n_time_steps, mode="train"):
        data_shape = features.shape
        batch_size = data_shape[0]
        channels = data_shape[-1]

        if mode == "train":
            if self.transform_args["static_gain_std"] > 0:
                warp_mat = torch.tile(
                    torch.unsqueeze(torch.eye(channels, device=self.device), dim=0),
                    (batch_size, 1, 1),
                )
                warp_mat += (
                    torch.randn_like(warp_mat) * self.transform_args["static_gain_std"]
                )
                features = torch.matmul(features, warp_mat)

            if self.transform_args["white_noise_std"] > 0:
                features += (
                    torch.randn(data_shape, device=self.device)
                    * self.transform_args["white_noise_std"]
                )

            if self.transform_args["constant_offset_std"] > 0:
                features += (
                    torch.randn((batch_size, 1, channels), device=self.device)
                    * self.transform_args["constant_offset_std"]
                )

            if self.transform_args["random_walk_std"] > 0:
                features += torch.cumsum(
                    torch.randn(data_shape, device=self.device)
                    * self.transform_args["random_walk_std"],
                    dim=self.transform_args["random_walk_axis"],
                )

            if self.transform_args["random_cut"] > 0:
                cut = np.random.randint(0, self.transform_args["random_cut"])
                features = features[:, cut:, :]
                n_time_steps = n_time_steps - cut

        if self.transform_args["smooth_data"]:
            features = gauss_smooth(
                inputs=features,
                device=self.device,
                smooth_kernel_std=self.transform_args["smooth_kernel_std"],
                smooth_kernel_size=self.transform_args["smooth_kernel_size"],
            )

        return features, n_time_steps

    def train(self):
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        self.logger.info(f"Starting training with {trainable:,} trainable parameters")

        train_losses = []
        val_losses = []
        val_PERs = []
        val_results = []

        val_steps_since_improvement = 0

        save_best_checkpoint = self.args.get("save_best_checkpoint", True)
        early_stopping = self.args.get("early_stopping", False)
        early_stopping_val_steps = self.args["early_stopping_val_steps"]

        train_start_time = time.time()

        for i, batch in enumerate(self.train_loader):
            self.model.train()
            self.optimizer.zero_grad()

            start_time = time.time()

            features = batch["input_features"].to(self.device)
            labels = batch["seq_class_ids"].to(self.device)
            n_time_steps = batch["n_time_steps"].to(self.device)
            phone_seq_lens = batch["phone_seq_lens"].to(self.device)
            day_indicies = batch["day_indicies"].to(self.device)

            with torch.autocast(
                device_type="cuda", enabled=self.args["use_amp"], dtype=torch.bfloat16
            ):
                features, n_time_steps = self.transform_data(
                    features, n_time_steps, "train"
                )

                logits = self.model(features, day_indicies)
                B, S, V = logits.shape

                max_target_len = phone_seq_lens.max().item()
                if max_target_len > S:
                    pad_T = max_target_len - S
                    last_step = logits[:, -1:, :].expand(B, pad_T, V)
                    logits = torch.cat([logits, last_step], dim=1)
                    S = max_target_len

                input_lengths = torch.full(
                    (B,),
                    S,
                    dtype=torch.long,
                    device=logits.device,
                )

                per_ex_loss = self.ctc_loss(
                    log_probs=logits.log_softmax(2).permute(1, 0, 2),
                    targets=labels,
                    input_lengths=input_lengths,
                    target_lengths=phone_seq_lens,
                )

                finite_mask = torch.isfinite(per_ex_loss)
                if not finite_mask.any():
                    self.logger.warning(
                        f"All CTC losses non-finite in train batch {i}. Skipping this batch."
                    )
                    continue

                loss = per_ex_loss[finite_mask].mean()

            loss.backward()

            grad_norm = None
            if self.args["grad_norm_clip_value"] > 0:
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    max_norm=self.args["grad_norm_clip_value"],
                    error_if_nonfinite=True,
                    foreach=True,
                )

            self.optimizer.step()
            self.learning_rate_scheduler.step()

            train_step_duration = time.time() - start_time
            train_losses.append(loss.detach().item())

            if i % self.args["batches_per_train_log"] == 0:
                self.logger.info(
                    f"Train batch {i}: loss={loss.detach().item():.3f} "
                    f"grad_norm={grad_norm if grad_norm is not None else 0:.2f} "
                    f"time={train_step_duration:.3f}"
                )

            if i % self.args["batches_per_val_step"] == 0 or i == (
                self.args["num_training_batches"] - 1
            ):
                self.logger.info(f"Running validation after training batch {i}")
                start_time = time.time()
                val_metrics = self.validation(
                    loader=self.val_loader,
                    return_logits=self.args["save_val_logits"],
                    return_data=self.args["save_val_data"],
                )
                val_step_duration = time.time() - start_time

                self.logger.info(
                    f"Val batch {i}: PER={val_metrics['avg_PER']:.4f} "
                    f"CTC loss={val_metrics['avg_loss']:.4f} time={val_step_duration:.3f}"
                )

                if self.args["log_individual_day_val_PER"]:
                    for d in val_metrics["day_PERs"].keys():
                        self.logger.info(
                            f"{self.args['dataset']['sessions'][d]} val PER: "
                            f"{val_metrics['day_PERs'][d]['total_edit_distance'] / val_metrics['day_PERs'][d]['total_seq_length']:.4f}"
                        )

                val_PERs.append(val_metrics["avg_PER"])
                val_losses.append(val_metrics["avg_loss"])
                val_results.append(val_metrics)

                new_best = False
                if val_metrics["avg_PER"] < self.best_val_PER:
                    self.logger.info(
                        f"New best val PER {self.best_val_PER:.4f} → {val_metrics['avg_PER']:.4f}"
                    )
                    self.best_val_PER = val_metrics["avg_PER"]
                    self.best_val_loss = val_metrics["avg_loss"]
                    new_best = True
                elif (
                    val_metrics["avg_PER"] == self.best_val_PER
                    and val_metrics["avg_loss"] < self.best_val_loss
                ):
                    self.logger.info(
                        f"New best val loss {self.best_val_loss:.4f} → {val_metrics['avg_loss']:.4f}"
                    )
                    self.best_val_loss = val_metrics["avg_loss"]
                    new_best = True

                if new_best:
                    if save_best_checkpoint:
                        self.logger.info("Checkpointing best model")
                        self.save_model_checkpoint(
                            os.path.join(
                                self.args["checkpoint_dir"], "best_checkpoint"
                            ),
                            self.best_val_PER,
                            self.best_val_loss,
                        )

                    if self.args["save_val_metrics"]:
                        with open(
                            os.path.join(
                                self.args["checkpoint_dir"], "val_metrics.pkl"
                            ),
                            "wb",
                        ) as f:
                            pickle.dump(val_metrics, f)

                    val_steps_since_improvement = 0
                else:
                    val_steps_since_improvement += 1

                if self.args["save_all_val_steps"]:
                    ckpt_path = os.path.join(
                        self.args["checkpoint_dir"], f"checkpoint_batch_{i}"
                    )
                    self.save_model_checkpoint(
                        ckpt_path, val_metrics["avg_PER"], val_metrics["avg_loss"]
                    )

                if early_stopping and (
                    val_steps_since_improvement >= early_stopping_val_steps
                ):
                    self.logger.info(
                        f"Val PER has not improved in {early_stopping_val_steps} validation steps. "
                        f"Stopping early at batch {i}."
                    )
                    break

        training_duration = time.time() - train_start_time
        self.logger.info(f"Best avg val PER achieved: {self.best_val_PER:.5f}")
        self.logger.info(f"Total training time: {training_duration / 60:.2f} minutes")

        if self.args["save_final_model"]:
            final_ckpt = os.path.join(
                self.args["checkpoint_dir"], f"final_checkpoint_batch_{i}"
            )
            self.save_model_checkpoint(final_ckpt, val_PERs[-1], val_losses[-1])

        train_stats = {
            "train_losses": train_losses,
            "val_losses": val_losses,
            "val_PERs": val_PERs,
            "val_metrics": val_results,
        }
        return train_stats

    def validation(self, loader, return_logits=False, return_data=False):
        self.model.eval()
        metrics = {}

        if return_logits:
            metrics["logits"] = []
            metrics["n_time_steps"] = []

        if return_data:
            metrics["input_features"] = []

        metrics["decoded_seqs"] = []
        metrics["true_seq"] = []
        metrics["phone_seq_lens"] = []
        metrics["transcription"] = []
        metrics["losses"] = []
        metrics["block_nums"] = []
        metrics["trial_nums"] = []
        metrics["day_indicies"] = []

        total_edit_distance = 0
        total_seq_length = 0

        day_per = {}
        for d in range(len(self.args["dataset"]["sessions"])):
            if self.args["dataset"]["dataset_probability_val"][d] == 1:
                day_per[d] = {"total_edit_distance": 0, "total_seq_length": 0}

        for i, batch in enumerate(loader):
            features = batch["input_features"].to(self.device)
            labels = batch["seq_class_ids"].to(self.device)
            n_time_steps = batch["n_time_steps"].to(self.device)
            phone_seq_lens = batch["phone_seq_lens"].to(self.device)
            day_indicies = batch["day_indicies"].to(self.device)

            day = day_indicies[0].item()
            if self.args["dataset"]["dataset_probability_val"][day] == 0:
                continue

            with torch.no_grad():
                with torch.autocast(
                    device_type="cuda",
                    enabled=self.args["use_amp"],
                    dtype=torch.bfloat16,
                ):
                    features, n_time_steps = self.transform_data(
                        features, n_time_steps, "val"
                    )

                    logits = self.model(features, day_indicies)
                    B, S, V = logits.shape

                    max_target_len = phone_seq_lens.max().item()
                    if max_target_len > S:
                        pad_T = max_target_len - S
                        last_step = logits[:, -1:, :].expand(B, pad_T, V)
                        logits = torch.cat([logits, last_step], dim=1)
                        S = max_target_len

                    input_lengths = torch.full(
                        (B,),
                        S,
                        dtype=torch.long,
                        device=logits.device,
                    )

                    per_ex_loss = self.ctc_loss(
                        torch.permute(logits.log_softmax(2), [1, 0, 2]),
                        labels,
                        input_lengths,
                        phone_seq_lens,
                    )

                    finite_mask = torch.isfinite(per_ex_loss)
                    if not finite_mask.any():
                        self.logger.warning(
                            f"All CTC losses non-finite in val batch {i}. Skipping this batch."
                        )
                        continue

                    loss = per_ex_loss[finite_mask].mean()

                metrics["losses"].append(loss.cpu().detach().numpy())

                batch_edit_distance = 0
                decoded_seqs = []
                for b in range(logits.shape[0]):
                    T = input_lengths[b].item()
                    decoded_seq = torch.argmax(logits[b, :T, :], dim=-1)
                    decoded_seq = torch.unique_consecutive(decoded_seq, dim=-1)
                    decoded_seq = decoded_seq.cpu().detach().numpy()
                    decoded_seq = np.array([idx for idx in decoded_seq if idx != 0])

                    true_seq = np.array(labels[b][0 : phone_seq_lens[b]].cpu().detach())
                    batch_edit_distance += taF.edit_distance(decoded_seq, true_seq)
                    decoded_seqs.append(decoded_seq)

            day_per[day]["total_edit_distance"] += batch_edit_distance
            day_per[day]["total_seq_length"] += torch.sum(phone_seq_lens).item()

            total_edit_distance += batch_edit_distance
            total_seq_length += torch.sum(phone_seq_lens)

            if return_logits:
                metrics["logits"].append(logits.cpu().float().numpy())
                metrics["n_time_steps"].append(input_lengths.cpu().numpy())

            if return_data:
                metrics["input_features"].append(batch["input_features"].cpu().numpy())

            metrics["decoded_seqs"].append(decoded_seqs)
            metrics["true_seq"].append(batch["seq_class_ids"].cpu().numpy())
            metrics["phone_seq_lens"].append(batch["phone_seq_lens"].cpu().numpy())
            metrics["transcription"].append(batch["transcriptions"].cpu().numpy())
            metrics["block_nums"].append(batch["block_nums"].numpy())
            metrics["trial_nums"].append(batch["trial_nums"].numpy())
            metrics["day_indicies"].append(batch["day_indicies"].cpu().numpy())

        avg_PER = (total_edit_distance / total_seq_length).item()

        metrics["day_PERs"] = day_per
        metrics["avg_PER"] = avg_PER
        metrics["avg_loss"] = float(np.mean(metrics["losses"])) if len(metrics["losses"]) > 0 else float("inf")

        return metrics

    def evaluate_full_validation(self):
        val_metrics = self.validation(
            loader=self.val_loader,
            return_logits=False,
            return_data=False,
        )

        self.logger.info("\n=== Full validation results from current model ===")
        self.logger.info(f"Average CTC loss: {val_metrics['avg_loss']:.4f}")
        self.logger.info(f"Average PER:      {val_metrics['avg_PER']:.4f}")

        for d, stats in val_metrics["day_PERs"].items():
            if stats["total_seq_length"] > 0:
                day_per = stats["total_edit_distance"] / stats["total_seq_length"]
                day_name = self.args["dataset"]["sessions"][d]
                self.logger.info(f"  {day_name}: PER = {day_per:.4f}")

        return val_metrics



In [7]:

args = {
    "mode": "train",
    "output_dir": None,
    "checkpoint_dir": None,

    "save_best_checkpoint": True,
    "save_all_val_steps": False,
    "save_final_model": True,

    "gpu_number": 0,
    "seed": 0,

    "use_torch_compile": False,
    "use_amp": True,

    "beta0": 0.9,
    "beta1": 0.999,
    "epsilon": 1e-8,
    "weight_decay": 1e-4,
    "weight_decay_day": 1e-4,
    "lr_max": 1e-4,
    "lr_min": 1e-6,
    "lr_max_day": 1e-4,
    "lr_min_day": 1e-6,
    "lr_decay_steps": 4000,
    "lr_decay_steps_day": 4000,
    "lr_warmup_steps": 500,
    "lr_warmup_steps_day": 500,
    "lr_scheduler_type": "cosine",

    "num_training_batches": 5000,
    "grad_norm_clip_value": 5.0,
    "batches_per_train_log": 100,
    "batches_per_val_step": 100,

    "early_stopping": False,
    "early_stopping_val_steps": 10,

    "save_val_logits": False,
    "save_val_data": False,
    "save_val_metrics": True,
    "log_individual_day_val_PER": False,

    "init_from_checkpoint": False,
    "init_checkpoint_path": "",

    "dataset": {
        "dataset_dir": None,
        "sessions": None,
        "n_classes": 42,
        "seed": 0,
        "temporal_bin": 50,
        "days_per_batch": 2,
        "batch_size": 64,
        "loader_shuffle": True,
        "num_dataloader_workers": 2,
        "dataset_probability_val": None,
        "feature_subset": None,
        "bad_trials_dict": None,
        "data_transforms": {
            "static_gain_std": 0.0,
            "white_noise_std": 0.0,
            "constant_offset_std": 0.0,
            "random_walk_std": 0.0,
            "random_walk_axis": 1,
            "random_cut": 0,
            "smooth_data": True,
            "smooth_kernel_std": 2.0,
            "smooth_kernel_size": 100,
        },
    },

    "model": {
        "n_input_features": None,
        "n_units": 256,
        "n_layers": 2,
        "patch_size": 0,
        "patch_stride": 0,
        "rnn_dropout": 0.1,
        "input_network": {
            "input_layer_dropout": 0.1,
            "input_trainable": True,
        },
        "transformer_name": "google/flan-t5-base",
        "rnn_trainable": True,
    },
}

PROJECT_ROOT = "/content/drive/MyDrive/STOR566Project"

DATA_ROOT = os.path.join(
    PROJECT_ROOT,
    "data",
    "brain-to-text-25",
    "t15_copyTask_neuralData",
    "hdf5_data_final",
)
sessions = [
    "t15.2023.08.11",
    "t15.2023.08.13",
    "t15.2023.08.18",
    "t15.2023.08.20",
    "t15.2023.08.25",
    "t15.2023.08.27",
    "t15.2023.09.01",
    "t15.2023.09.03",
    "t15.2023.09.24",
    "t15.2023.09.29",
    "t15.2023.10.01",
    "t15.2023.10.06",
    "t15.2023.10.08",
    "t15.2023.10.13",
    "t15.2023.10.15",
    "t15.2023.10.20",
    "t15.2023.10.22",
    "t15.2023.11.03",
    "t15.2023.11.04",
    "t15.2023.11.17",
    "t15.2023.11.19",
    "t15.2023.11.26",
    "t15.2023.12.03",
    "t15.2023.12.08",
    "t15.2023.12.10",
    "t15.2023.12.17",
    "t15.2023.12.29",
    "t15.2024.02.25",
    "t15.2024.03.03",
    "t15.2024.03.08",
    "t15.2024.03.15",
    "t15.2024.03.17",
    "t15.2024.04.25",
    "t15.2024.04.28",
    "t15.2024.05.10",
    "t15.2024.06.14",
    "t15.2024.07.19",
    "t15.2024.07.21",
    "t15.2024.07.28",
    "t15.2025.01.10",
    "t15.2025.01.12",
    "t15.2025.03.14",
    "t15.2025.03.16",
    "t15.2025.03.30",
    "t15.2025.04.13",
]

# Number of neural input features (from YAML: 512 = 256 electrodes * 2 features)
n_input_features = 512
MY_OUT_ROOT = "/content/cnn_t5_runs_local"
os.makedirs(MY_OUT_ROOT, exist_ok=True)
args["output_dir"] = f"{MY_OUT_ROOT}/outputs"
args["checkpoint_dir"] = f"{MY_OUT_ROOT}/checkpoints"
os.makedirs(args["output_dir"], exist_ok=True)
os.makedirs(args["checkpoint_dir"], exist_ok=True)

# fill dataset
args["dataset"]["dataset_dir"] = DATA_ROOT
args["dataset"]["sessions"] = sessions
args["dataset"]["dataset_probability_val"] = [1 for _ in sessions]
args["model"]["n_input_features"] = n_input_features

trainer = CNN_Transformer_Trainer(args)
train_stats = trainer.train()

2025-11-20 16:07:31,451: Using device: cuda:0


INFO:BrainToTextTrainer:Using device: cuda:0
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

2025-11-20 16:07:41,117: Model has 247,855,738 parameters


INFO:BrainToTextTrainer:Model has 247,855,738 parameters


2025-11-20 16:07:41,120: Model has 34,560 day-specific parameters | 0.01% of total parameters


INFO:BrainToTextTrainer:Model has 34,560 day-specific parameters | 0.01% of total parameters


KeyboardInterrupt: 